# SETUP FOR BETTER GPU

1) Install UCSD VPN: https://blink.ucsd.edu/technology/network/connections/off-campus/VPN/
2) ssh on local machine to ssh username@dsmlp-login.ucsd.edu where username is <username>@ucsd.edu
3) Run the following command on the ssh: launch.sh -c 8 -m 32 -g 1 -v a5000
4) Open the link it provides
5) Make sure to **shut down** the pod when you're done from File > Shut Down. And then exit ssh from your terminal

6) If it says GPU already in use (1 of 1 allocated): run the following "kubectl get pods" and "kubectl delete pod <paste-pod-name-here>"


## Starter Code for Datahub

### IF THIS IS YOUR FIRST TIME RUNNING THIS ON DATA HUB
You need to run the cell immediately below so that you can install/setup the kernel. Its called "Python (cse151b)". You should **always** select this kernel in the top right of jupyter. 

### If you're coming back, make sure you re-run everything from the next cell (imports) and further down

The purpose of this starter code is so that you can basically duplicate this and run your experiments in a separate notebook.

# START HERE IF ITS YOUR FIRST TIME HERE

In [ ]:
# =========================================================
# DATAHUB SETUP & IMPORTS. (May take a while!) Comment me out after first run!
# =========================================================

# remove old kernel
!jupyter kernelspec uninstall -y cse151b

# Remove old .venv
!rm -rf .venv

# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

print("Old environment and kernel destroyed.")

# Create a virtual environment
!~/.local/bin/uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install sympy numpy transformers vllm==0.19.1 tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter scikit-learn pandas

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding. Then click on current kernel -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'. Restart kernel again. Comment out this code!")

# START HERE IF YOU'VE ALREADY SETUP! 

# MAKE SURE TO RESTART KERNEL BEFORE CONTINUING!

## Library Setup

In [1]:
!source ./.venv/bin/activate

import json
import os
import random

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 8192

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

import pandas as pd
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm
from datetime import datetime 
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

## Config + Data Loading
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [2]:
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0" 
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/datahub_experiments.jsonl"
MAX_TOKENS  = 8192

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## Hyperparam Sandbox + LLM Setup

If this doesn't work, what worked for me was:
1) shut down ALL OTHER kernels in the directory
2) re-run the first cell (commented out one)

If it still fails:
1) run "pkill -9 -f python" in terminal. This will kill the datahub pod and (most likely) wipe /dev/shm
2) re-launch the container
3) ensure all other kernels are shut down and go from the top.

In [3]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token

# llm = LLM(
#     model=MODEL_ID,
#     quantization="bitsandbytes",
#     load_format="bitsandbytes",
#     enable_prefix_caching=False,
#     enforce_eager=True,
#     gpu_memory_utilization=0.85,
#     max_model_len=16384,
#     trust_remote_code=True,
#     max_num_seqs=256,
#     max_num_batched_tokens=32768,
# )

# sampling_params = SamplingParams(
#     max_tokens=MAX_TOKENS,
#     temperature=0.6,
#     top_p=0.95,
#     top_k=20,
#     min_p=0.0,
#     presence_penalty=0.0,
#     repetition_penalty=1.0,
# )

# print("Model and parameters loaded successfully in Eager Mode.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16", 
    quantization="bitsandbytes", 
    load_format="auto",
    enable_prefix_caching=True, 
    enforce_eager=True,        
    gpu_memory_utilization=0.95, 
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    presence_penalty=0.0,
    repetition_penalty=1.05, # slight penalty for repetition
)

print("Model loaded with CUDAGraph + BF16.")

INFO 05-02 00:48:09 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 16384, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.95, 'max_num_batched_tokens': 32768, 'max_num_seqs': 256, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'enforce_eager': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


INFO 05-02 00:48:10 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 05-02 00:48:10 [model.py:1678] Using max model len 16384
INFO 05-02 00:48:10 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=32768.
INFO 05-02 00:48:12 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-02 00:48:12 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-02 00:48:12 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 05-02 00:48:12 [vllm.py:1025] Cudagraph is disabled under eager mode
INFO 05-02 00:48:12 [compilation.py:292] Enabled custom fusions: norm_quant, act_quant
(EngineCore pid=7445) INFO 05-02 00:48:13 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwe

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=7445) INFO 05-02 00:48:18 [gpu_model_runner.py:4820] Model loading took 2.7 GiB memory and 3.602572 seconds
(EngineCore pid=7445) INFO 05-02 00:48:23 [gpu_worker.py:436] Available KV cache memory: 17.11 GiB
(EngineCore pid=7445) INFO 05-02 00:48:23 [kv_cache_utils.py:1319] GPU KV cache size: 124,560 tokens
(EngineCore pid=7445) INFO 05-02 00:48:23 [kv_cache_utils.py:1324] Maximum concurrency for 16,384 tokens per request: 7.60x
(EngineCore pid=7445) INFO 05-02 00:48:23 [core.py:283] init engine (profile, create kv cache, warmup model) took 4.77 seconds
(EngineCore pid=7445) WARNING 05-02 00:48:24 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=7445) WARNING 05-02 00:48:24 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=7445) INFO 05-02 00:48:

## Experiment/Submission Config - CONFIGURE ME!

In [12]:
IS_EXPERIMENT   = True               # Set to False to run the full 1126 dataset
EXPERIMENT_NAME = "few_shot_prompting"           # Only used if IS_EXPERIMENT is True
SAVE_EVAL       = True               # Set to False for private test set (no ground truth)

# Dynamic Naming Logic
date_str = datetime.now().strftime("%Y-%m-%d")

if IS_EXPERIMENT:
    output_filename = f"experiment_{EXPERIMENT_NAME}_{date_str}.jsonl"
else:
    output_filename = f"submission_{date_str}.jsonl"

OUTPUT_PATH = Path("results") / output_filename
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"RUN MODE : {'EXPERIMENT (50 samples)' if IS_EXPERIMENT else 'FULL DATASET (1126 items)'}")
print(f"SAVE TO  : {OUTPUT_PATH}")

RUN MODE : EXPERIMENT (50 samples)
SAVE TO  : results/experiment_few_shot_prompting_2026-05-02.jsonl


## Prompt Engineering Sandbox

### No few shot prompting

In [5]:
# SYSTEM_PROMPT_MATH = (
#     "You are an expert mathematician. Solve the problem step-by-step. "
#     "Put your final answer inside \\boxed{}. "
#     "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
#     "e.g. \\boxed{3, 7}."
# )

# SYSTEM_PROMPT_MCQ = (
#     "You are an expert mathematician. "
#     "Read the problem and the answer choices below, then select the single best answer. "
#     "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
# )

# def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
#     if options:
#         labels    = [chr(65 + i) for i in range(len(options))]
#         opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
#         return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
#     return SYSTEM_PROMPT_MATH, question

### Few Shot Prompting

In [13]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices carefully. "
    "Think through the problem step by step, then select the single best answer. "
    "At the very end of your response, output the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

FEW_SHOT_EXAMPLES = [
    # ---------------------------------------------------------
    # MCQ 1: Simple (Probability)
    # Teaches the model to list out small sample spaces and simplify fractions.
    # ---------------------------------------------------------
    {
        "type": "mcq",
        "question": "A fair six-sided die is rolled twice. What is the probability that the sum of the two rolls is exactly 7?",
        "options": ["1/6", "1/36", "1/12", "1/4", "7/36"],
        "reasoning": "The total number of possible outcomes when rolling a six-sided die twice is 6 * 6 = 36.\nWe need to find the number of outcomes where the sum is 7. Let's list the valid pairs: (1,6), (2,5), (3,4), (4,3), (5,2), and (6,1).\nCounting them, there are exactly 6 such pairs.\nTherefore, the probability is 6 / 36.\nReducing this fraction gives 1/6.\nLooking at the given options, 1/6 corresponds to option A.",
        "final_answer": "A"
    },
    
    # ---------------------------------------------------------
    # MCQ 2: Medium (Calculus)
    # Teaches the model to use variable substitution and evaluate bounds.
    # ---------------------------------------------------------
    {
        "type": "mcq",
        "question": "Evaluate the definite integral $int_0^{\\pi/2} \\sin(x)\\cos(x) dx$.",
        "options": ["1", "0", "1/2", "\\pi/4", "-1/2"],
        "reasoning": "To evaluate this integral, I can use u-substitution.\nLet u = \\sin(x). Then the derivative du = \\cos(x) dx.\nNext, I need to change the bounds of integration. \nWhen x = 0, u = \\sin(0) = 0.\nWhen x = \\pi/2, u = \\sin(\\pi/2) = 1.\nThe integral transforms to: $int_0^1 u du$.\nThe antiderivative of u is u^2 / 2.\nEvaluating this from 0 to 1 gives: (1^2 / 2) - (0^2 / 2) = 1/2 - 0 = 1/2.\nOption C is 1/2.",
        "final_answer": "C"
    },

    # ---------------------------------------------------------
    # MCQ 3: Complex (Linear Algebra)
    # Teaches the model to apply theorems (eigenvalue mapping) to avoid brute force.
    # ---------------------------------------------------------
    {
        "type": "mcq",
        "question": "Let A be a 3x3 matrix with eigenvalues 1, -2, and 4. What is the determinant of $A^2 + 2I$?",
        "options": ["18", "54", "108", "216", "324"],
        "reasoning": "There is a property in linear algebra that if \\lambda is an eigenvalue of A, then f(\\lambda) is an eigenvalue of f(A).\nHere, our function is f(A) = A^2 + 2I. Therefore, the new eigenvalues will be \\lambda^2 + 2.\nLet's calculate the new eigenvalues based on the original ones (1, -2, 4):\nFor \\lambda_1 = 1: (1)^2 + 2 = 1 + 2 = 3.\nFor \\lambda_2 = -2: (-2)^2 + 2 = 4 + 2 = 6.\nFor \\lambda_3 = 4: (4)^2 + 2 = 16 + 2 = 18.\nThe determinant of a matrix is equal to the product of its eigenvalues.\nSo, \\det(A^2 + 2I) = 3 * 6 * 18.\nFirst, 3 * 6 = 18.\nThen, 18 * 18 = 324.\nChecking the options, 324 matches option E.",
        "final_answer": "E"
    },

    # ---------------------------------------------------------
    # Free-Form 1: Simple (Arithmetic/Sequences)
    # Teaches the model to recognize known formulas rather than brute-force counting.
    # ---------------------------------------------------------
    {
        "type": "free-form",
        "question": "Find the sum of the first 50 positive odd integers.",
        "options": None,
        "reasoning": "The sequence of positive odd integers is 1, 3, 5, 7, ...\nThis is an arithmetic sequence where the first term a = 1 and the common difference d = 2.\nThere is a well-known mathematical property that the sum of the first n positive odd integers is exactly n^2.\nHere, n = 50.\nTherefore, the sum is 50^2.\n50 * 50 = 2500.\nTo double check, I can use the arithmetic sum formula: S_n = (n/2)(2a + (n-1)d).\nS_50 = (50/2)(2(1) + 49(2)) = 25(2 + 98) = 25(100) = 2500.\nBoth methods confirm the answer is 2500.",
        "final_answer": "2500"
    },

    # ---------------------------------------------------------
    # Free-Form 2: Medium (Combinatorics)
    # Teaches the model to handle overlapping states / duplicate items in counting.
    # ---------------------------------------------------------
    {
        "type": "free-form",
        "question": "How many distinct 5-letter words can be formed using the letters from the word 'APPLE'?",
        "options": None,
        "reasoning": "The word 'APPLE' consists of 5 letters. \nIf all the letters were entirely unique, the number of permutations would simply be 5! (5 factorial).\nHowever, the letter 'P' appears exactly 2 times. The other letters ('A', 'L', 'E') appear exactly 1 time each.\nTo find the number of distinct permutations, we must divide the total factorial by the factorial of the frequencies of the repeated characters to remove duplicates.\nThe formula is n! / (n_1! * n_2! * ...).\nHere, n = 5, and the frequency of 'P' is 2.\nSo, the number of distinct arrangements is 5! / 2!.\n5! = 5 * 4 * 3 * 2 * 1 = 120.\n2! = 2 * 1 = 2.\n120 / 2 = 60.\nThere are 60 distinct words.",
        "final_answer": "60"
    },

    # ---------------------------------------------------------
    # Free-Form 3: Complex (Number Theory)
    # Teaches the model to use modular arithmetic theorems to reduce massive numbers.
    # ---------------------------------------------------------
    {
        "type": "free-form",
        "question": "What is the remainder when $3^{2026}$ is divided by 13?",
        "options": None,
        "reasoning": "We need to compute 3^{2026} \\pmod{13}.\nBecause 13 is a prime number and 3 is not divisible by 13, we can use Fermat's Little Theorem, which states that a^{p-1} \\equiv 1 \\pmod p.\nHere, a = 3 and p = 13. Therefore, 3^{12} \\equiv 1 \\pmod{13}.\nTo simplify 3^{2026}, we need to divide the exponent 2026 by 12 and find the remainder.\nLet's do the division: 2026 / 12.\n12 * 100 = 1200. Remaining: 826.\n12 * 60 = 720. Remaining: 106.\n12 * 8 = 96. Remaining: 10.\nSo, 2026 = 12 * 168 + 10. The remainder is 10.\nThis means 3^{2026} = (3^{12})^{168} * 3^{10} \\equiv 1^{168} * 3^{10} \\equiv 3^{10} \\pmod{13}.\nNow we just need to compute 3^{10} \\pmod{13}.\nLet's find smaller powers of 3 modulo 13:\n3^1 = 3\n3^2 = 9\n3^3 = 27 = 2(13) + 1 \\equiv 1 \\pmod{13}.\nThis is a massive shortcut! Since 3^3 \\equiv 1 \\pmod{13}, we can rewrite 3^{10} in terms of 3^3.\n3^{10} = (3^3)^3 * 3^1 \\equiv (1)^3 * 3 \\equiv 1 * 3 = 3 \\pmod{13}.\nThe remainder is 3.",
        "final_answer": "3"
    }
]

def build_chat_messages(target_question: str, target_options: list | None, examples: list) -> list:
    is_target_mcq = bool(target_options)
    
    system_prompt = SYSTEM_PROMPT_MCQ if is_target_mcq else SYSTEM_PROMPT_MATH
    messages = [{"role": "system", "content": system_prompt}]
    
    for ex in examples:
        if ex["options"]:
            labels = [chr(65 + i) for i in range(len(ex["options"]))]
            opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, ex["options"]))
            user_content = f"{ex['question']}\n\nOptions:\n{opts_text}"
        else:
            user_content = ex["question"]
            
        assistant_content = f"<think>\n{ex['reasoning']}\n</think>\n\\boxed{{{ex['final_answer']}}}"
        
        messages.append({"role": "user", "content": user_content})
        messages.append({"role": "assistant", "content": assistant_content})
    
    if is_target_mcq:
        labels = [chr(65 + i) for i in range(len(target_options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, target_options))
        target_content = f"{target_question}\n\nOptions:\n{opts_text}"
    else:
        target_content = target_question
        
    messages.append({"role": "user", "content": target_content})
    
    return messages

## Dataset Generation

In [14]:
if IS_EXPERIMENT:
    SUBSET_SIZE = 50
    RANDOM_SEED = 42
    random.seed(RANDOM_SEED)

    mcq_pool  = [d for d in data if d.get("options")]
    free_pool = [d for d in data if not d.get("options")]

    mcq_ratio = len(mcq_pool) / len(data)
    n_mcq  = max(1, round(SUBSET_SIZE * mcq_ratio))
    n_free = SUBSET_SIZE - n_mcq

    data_to_run = random.sample(mcq_pool, n_mcq) + random.sample(free_pool, n_free)
    random.shuffle(data_to_run)
else:
    data_to_run = data

print(f"Data ready. Queuing {len(data_to_run)} questions for inference.")

Data ready. Queuing 50 questions for inference.


## Inference 

### No few shot prompting

In [7]:
# prompts = []
# for item in data_to_run:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# print(f"Generating responses for {len(prompts)} questions...")

# # Chunking to prevent EngineDead errors
# BATCH_SIZE = 8   
# all_outputs = []

# for i in range(0, len(prompts), BATCH_SIZE):
#     batch = prompts[i:i + BATCH_SIZE]
#     print(f"Processing batch {i//BATCH_SIZE + 1} / {(len(prompts) + BATCH_SIZE - 1) // BATCH_SIZE}")
#     outputs = llm.generate(batch, sampling_params=sampling_params)
#     all_outputs.extend(outputs)

# responses = [out.outputs[0].text.strip() for out in all_outputs]

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data_to_run[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 50 questions...
Processing batch 1 / 7


Rendering prompts:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processing batch 2 / 7


Rendering prompts:   0%|          | 0/8 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processing batch 7 / 7


Rendering prompts:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=910) ──
Okay, let's see. So the problem is about three video editors working together on adding special effects. We need to find out how long it would take them all together. Hmm, this sounds like a work rate problem. Let me recall how those work. 

First, when dealing with work rates, the idea is that each person's rate is how much of the job they can do per hour. Then, when they work together, their rat ...

── Response 1 (id=308) ──
Okay, let's tackle this problem step by step. The question is asking whether each of the given functions is a power function. First, I need to remember what a power function is. From what I recall, a power function is generally defined as a function of the form \( y = kx^a \), where \( k \) and \( a \) are constants, and \( x \) is the variable. So, it's a single term with a constant coefficient m ...

── Response 2 (id=54) ──
Okay, let's try to figure out this problem step by step. So, first, let me make sure I understand the problem 

### Few-shot prompting

In [15]:
prompts = []
for item in data_to_run:
    # Build few-shot chat history
    messages = build_chat_messages(
        target_question=item["question"], 
        target_options=item.get("options"), 
        examples=FEW_SHOT_EXAMPLES
    )
    
    # Apply the template to messages
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

print(f"Generating responses for {len(prompts)} questions...")

# Chunking to prevent EngineDead errors
BATCH_SIZE = 8   
all_outputs = []

for i in range(0, len(prompts), BATCH_SIZE):
    batch = prompts[i:i + BATCH_SIZE]
    print(f"Processing batch {i//BATCH_SIZE + 1} / {(len(prompts) + BATCH_SIZE - 1) // BATCH_SIZE}")
    outputs = llm.generate(batch, sampling_params=sampling_params)
    all_outputs.extend(outputs)

# Extract raw responses
responses = [out.outputs[0].text.strip() for out in all_outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data_to_run[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 50 questions...
Processing batch 1 / 7


Rendering prompts:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processing batch 2 / 7


Rendering prompts:   0%|          | 0/8 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processing batch 7 / 7


Rendering prompts:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=910) ──
Okay, let's try to figure out this problem step by step. So, we have three video editors with different times to add special effects: Level III takes 18 hours, Level II takes 26 hours, and Level I takes 34 hours. We need to find how long it would take them all working together. 

First, I remember that when dealing with rates (work rates), the idea is that each person's rate is their work done per ...

── Response 1 (id=308) ──
Okay, let's tackle each part one by one. First, I need to remember what a power function is. From what I recall, a power function is a function of the form \( y = kx^n \), where \( k \) is a constant and \( n \) is a real number. So it's a single term with a variable raised to a power multiplied by a constant coefficient. 

Starting with question 1: \( y = 3x^3 + 2x^2 \). This is a polynomial with ...

── Response 2 (id=54) ──
Okay, let's try to solve this problem step by step. First, let's make sure I understand the problem correctly.

## Score Responses + Summary

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [16]:
# Load Judger
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m: return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

results = []
for item, response in tqdm(zip(data_to_run, responses), total=len(data_to_run), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = extract_letter(response) == str(gold).strip().upper()
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list))
        except Exception:
            correct = False

    results.append({
        "id": item.get("id"), "is_mcq": is_mcq, "gold": gold,
        "response": response, "correct": correct,
    })

# --- PANDAS METRICS TABLE ---
def generate_metrics_df(results):
    rows = []
    for subset_name, is_mcq_flag in [("Overall", None), ("MCQ", True), ("Free-Form", False)]:
        if is_mcq_flag is None:
            subset = results
        else:
            subset = [r for r in results if r["is_mcq"] == is_mcq_flag]
            
        if not subset: continue
            
        y_true = [1] * len(subset)
        y_pred = [1 if r["correct"] else 0 for r in subset]
        
        rows.append({
            "Category": subset_name,
            "Accuracy": f"{accuracy_score(y_true, y_pred):.2%}",
            "Precision": round(precision_score(y_true, y_pred, zero_division=0), 3),
            "Recall": round(recall_score(y_true, y_pred, zero_division=0), 3),
            "F1 Score": round(f1_score(y_true, y_pred, zero_division=0), 3),
            "Correct": sum(y_pred),
            "Total": len(y_pred)
        })
    return pd.DataFrame(rows).set_index("Category")

metrics_df = generate_metrics_df(results)
display(metrics_df)

Scoring: 100%|██████████| 50/50 [00:01<00:00, 36.17it/s]


,Accuracy,Precision,Recall,F1 Score,Correct,Total
Category,,,,,,
Overall,52.00%,1.0,0.520,0.684,26,50
MCQ,58.82%,1.0,0.588,0.741,10,17
Free-Form,48.48%,1.0,0.485,0.653,16,33


## Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [17]:
with open(OUTPUT_PATH, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {OUTPUT_PATH}")

Saved 50 records to results/experiment_few_shot_prompting_2026-05-02.jsonl
